In [ ]:
import os
import time
from typing import List, Optional
from pinecone import Pinecone, ServerlessSpec
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import UnstructuredMarkdownLoader

class RAG_Service:
    def __init__(
            self,
            index_name: str,
            api_key: Optional[str] = None,
            model_name: str = "Qwen/Qwen3-VL-Embedding-2B",
            dimension: int = 1536, # Qwen2-1.5B 默认维度
            metric: str = "cosine",
            region: str = "us-east-1"
    ):
        self.index_name = index_name
        self.api_key = api_key or os.getenv("PINECONE_API_KEY")
        self.dimension = dimension
        self.metric = metric
        self.region = region

        # 1. 初始化 Pinecone 客户端
        self.pc = Pinecone(api_key=self.api_key)

        # 2. 初始化本地 Qwen Embedding 模型
        print(f"正在加载嵌入模型: {model_name}...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda'},
            encode_kwargs={'normalize_embeddings': True}
        )

        # 3. 初始化语义分块器
        self.text_splitter = SemanticChunker(self.embeddings, breakpoint_threshold_type="percentile")

        # 绑定索引操作对象
        if self.index_name in [idx.name for idx in self.pc.list_indexes()]:
            self.index = self.pc.Index(self.index_name)
        else:
            self.index = None

    def create_index(self):
        """创建索引，如果已存在则跳过"""
        if self.index_name not in [idx.name for idx in self.pc.list_indexes()]:
            print(f"正在创建索引: {self.index_name}...")
            self.pc.create_index(
                name=self.index_name,
                dimension=self.dimension,
                metric=self.metric,
                spec=ServerlessSpec(cloud="aws", region=self.region)
            )
            # 等待索引初始化完成
            while not self.pc.describe_index(self.index_name).status['ready']:
                time.sleep(1)

        self.index = self.pc.Index(self.index_name)
        print(f"索引 {self.index_name} 已就绪。")

    def add_Document(self, file_path: str, namespace: str = "default"):
        """载入 Markdown 文档，进行语义分块、向量化并上传"""
        if not self.index:
            raise ValueError("索引未初始化，请先调用 create_index()")

        print(f"正在处理文档: {file_path} ...")
        # 1. 加载 Markdown
        loader = UnstructuredMarkdownLoader(file_path)
        raw_docs = loader.load()

        # 2. 语义分块
        semantic_chunks = self.text_splitter.split_documents(raw_docs)

        # 3. 准备上传数据格式
        vectors_to_upsert = []
        for i, doc in enumerate(semantic_chunks):
            # 生成向量嵌入
            embedding = self.embeddings.embed_query(doc.page_content)

            # 这里的 ID 建议包含文件名以防重复
            doc_id = f"{os.path.basename(file_path)}_{i}"

            vectors_to_upsert.append({
                "id": doc_id,
                "values": embedding,
                "metadata": {
                    "text": doc.page_content, # 必须存入原文以便搜索返回
                    "source": file_path
                }
            })

            # 分批上传，防止 payload 过大
            if len(vectors_to_upsert) >= 100:
                self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)
                vectors_to_upsert = []

        if vectors_to_upsert:
            self.index.upsert(vectors=vectors_to_upsert, namespace=namespace)

        print(f"成功将 {len(semantic_chunks)} 个语义块上传至 Namespace: {namespace}")

    def Search_Vector(self, query: str, top_k: int = 3, namespace: str = "default"):
        """执行向量搜索并返回结果文本"""
        if not self.index:
            raise ValueError("索引未初始化")

        # 1. 向量化用户问题
        query_vector = self.embeddings.embed_query(query)

        # 2. 在 Pinecone 中检索
        search_results = self.index.query(
            vector=query_vector,
            top_k=top_k,
            namespace=namespace,
            include_metadata=True
        )

        # 3. 提取元数据中的文本内容
        context_list = []
        for res in search_results["matches"]:
            context_list.append({
                "text": res["metadata"]["text"],
                "score": res["score"],
                "source": res["metadata"]["source"]
            })

        return context_list

In [ ]:
import re
import os
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

class LawDocumentProcessor:
    def __init__(self):
        # 1. 设置按标题切分的规则（保留标题到元数据）
        headers_to_split_on = [
            ("#", "Header_1"),
            ("##", "Header_2"),
            ("###", "Header_3"),
        ]
        self.md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

        # 2. 设置递归切分规则（控制具体长度）
        # 假设 1 token ≈ 1.5 到 2 个汉字，200 token 约设为 400 字符
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=600,
            chunk_overlap=50,
            separators=["\n\n", "\n", "。", "；", " "], # 优先按段落切
            add_start_index=True
        )

    def extract_metadata_from_text(self, text, file_path):
        """利用正则从文本中提取案由和字号，从路径提取年份"""
        metadata = {}

        # 提取年份：假设文件名或路径包含 202X
        year_match = re.search(r"20\d{2}", file_path)
        metadata["year"] = year_match.group(0) if year_match else "Unknown"

        # 提取裁判书字号：匹配如（2023）最高法民终...号
        case_num_pattern = r"（\d{4}）[\u4e00-\u9fa5]+\d+号"
        case_num_match = re.search(case_num_pattern, text)
        metadata["case_number"] = case_num_match.group(0) if case_num_match else "未识别"

        # 提取案由：通常在字号之后，或者是特定的段落
        # 这里建议根据你的文档具体格式调整正则
        case_cause_pattern = r"案由[:：]\s*([\u4e00-\u9fa5]+)"
        cause_match = re.search(case_cause_pattern, text)
        metadata["case_cause"] = cause_match.group(1) if cause_match else "通用"

        return metadata

    def process_file(self, file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # A. 提取全局元数据
        global_metadata = self.extract_metadata_from_text(content, file_path)

        # B. 第一步：按标题切分（会自动把标题放入 metadata）
        initial_chunks = self.md_splitter.split_text(content)

        final_documents = []

        # C. 第二步：对每个标题块进行长度检查和二次切分
        for chunk in initial_chunks:
            # 合并全局元数据和标题元数据
            current_metadata = {**global_metadata, **chunk.metadata}

            # 如果该标题块太长，进行递归切分
            sub_chunks = self.text_splitter.split_text(chunk.page_content)

            for sub_text in sub_chunks:
                # 过滤掉太短的碎片（根据你的 200 token 要求）
                if len(sub_text) < 300: # 粗略估算
                    continue

                new_doc = Document(
                    page_content=sub_text,
                    metadata=current_metadata.copy()
                )
                final_documents.append(new_doc)

        return final_documents

# 使用示例
processor = LawDocumentProcessor()
all_docs = processor.process_file("path/to/your/2023_cases.md")
print(f"切分完成，共获得 {len(all_docs)} 个分块")
print(f"第一个分块元数据: {all_docs[0].metadata}")